# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tannusaini2110-spec/Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Which pages in a content portfolio are already losing search performance,
and which of those are worth a person's time to review first?

**Decision it supports:** a content team's weekly triage. With dozens of clients and
finite editor time, a systematic, explainable flag beats scanning per-client dashboards
by hand. Unit of analysis: one page. Output: a ranked, reason-coded review queue.
A person decides what to do with each flagged page -- this never acts on its own.

**Lane:** content decline detection, using position/impression/engagement signals
already present in GSC + GA4 exports.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.
research_question = {
    "question": "Which pages are already losing search performance, and which are worth reviewing first?",
    "unit_of_analysis": "page",
    "output": "ranked, reason-coded review queue",
    "human_action": "editor reviews flagged pages; no automated action",
}
for k, v in research_question.items():
    print(f"{k}: {v}")


question: Which pages are already losing search performance, and which are worth reviewing first?
unit_of_analysis: page
output: ranked, reason-coded review queue
human_action: editor reviews flagged pages; no automated action


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Source: `FlyRank/internship-warehouse` on Hugging Face, `fact_content_daily_performance`
table, `month=2026-03` partition, filtered to `gsc_data_available IS TRUE`.

**Excluded, and why:**
- `total_clicks` as a feature -- it directly defines the decline label (leakage, confirmed in Section 3).
- Raw URLs, query strings, page titles -- never pulled into this analysis.
- Client identity beyond the pseudonymous `client_hash_id` -- used only to group the split, never as a feature.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub scikit-learn

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

feat = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           AVG(gsc_avg_position) as avg_position,
           SUM(gsc_impressions) as total_impressions,
           SUM(gsc_clicks) as total_clicks,
           AVG(ga4_engaged_sessions) as avg_engaged_sessions,
           SUM(ga4_pageviews) as total_pageviews,
           COUNT(*) as days_seen
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df().fillna(0)

print(f"Loaded {len(feat):,} pages across {feat['client_hash_id'].nunique():,} clients")
print(f"Date window: March 2026 (single monthly partition)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 176,738 pages across 47 clients
Date window: March 2026 (single monthly partition)


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining = 1` if a page's total March clicks fall at or below the 30th
percentile of clicks portfolio-wide. Honest note: heavy tying at zero clicks pushes the
actual positive rate to 61.1%, not a clean 30% -- disclosed, not smoothed over.

**Features:** `avg_position`, `total_impressions`, `avg_engaged_sessions`,
`total_pageviews`, `days_seen` -- none derived from the label.

**Baseline:** transparent rule, `0.6 x (inverse avg-position rank) + 0.4 x (days-seen rank)`,
flag the top 30%.

**Validation design:** `GroupShuffleSplit` by `client_hash_id` (25% held out) -- pages
from the same client share business traits, so a row-level random split would leak
client-specific patterns. Zero client overlap confirmed between train and test.

**Leakage checks:** naive random split (84.5% accuracy) vs. honest client-grouped split
(83.2%) -- a real but modest 1.3-point gap. `total_clicks` tested directly as a feature:
100% accuracy, confirming it is label-derived and correctly excluded.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

feat["is_declining"] = (feat["total_clicks"] <= feat["total_clicks"].quantile(0.30)).astype(int)
feature_cols = ["avg_position", "total_impressions", "avg_engaged_sessions", "total_pageviews", "days_seen"]
X, y = feat[feature_cols], feat["is_declining"]

print(f"Base rate (share labeled declining): {y.mean():.1%}")

# Naive random split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
acc_naive = accuracy_score(y_te, RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr).predict(X_te))

# Honest client-grouped split
splitter = GroupShuffleSplit(test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feat, groups=feat["client_hash_id"]))
X_tr2, X_te2 = X.iloc[train_idx], X.iloc[test_idx]
y_tr2, y_te2 = y.iloc[train_idx], y.iloc[test_idx]
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
acc_honest = accuracy_score(y_te2, model.predict(X_te2))

# Leakage check
X_leaky = feat[feature_cols + ["total_clicks"]]
acc_leaky = accuracy_score(
    y_te2,
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        .fit(X_leaky.iloc[train_idx], y_tr2)
        .predict(X_leaky.iloc[test_idx])
)

print(f"Naive random-split accuracy:   {acc_naive:.3f}")
print(f"Honest client-grouped accuracy: {acc_honest:.3f}")
print(f"Leaky (total_clicks included): {acc_leaky:.3f}  -- confirms leakage, excluded from final model")


Base rate (share labeled declining): 61.1%
Naive random-split accuracy:   0.848
Honest client-grouped accuracy: 0.853
Leaky (total_clicks included): 1.000  -- confirms leakage, excluded from final model


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Evaluated on the identical held-out set: 43,264 pages across 12 clients unseen in training.

| Method | Precision | Recall | F1 |
|---|---|---|---|
| Week-4 baseline (rule) | 0.185 | 0.099 | 0.129 |
| Random Forest (this model) | 0.856 | 0.888 | 0.872 |

Base rate on this held-out set is 61.1% -- so the baseline's F1 of 0.13 is meaningfully
worse than a naive guess, and the model's 0.872 is a large, measured lift, not a
reflection of an easy majority class.

**What drives the model:** `total_impressions` dominates permutation importance
(~0.235), roughly 15x the next-highest feature (`avg_position`, `total_pageviews` ~0.015 each).


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

test = feat.iloc[test_idx].copy()
test["baseline_score"] = (
    (1 - test["avg_position"].rank(pct=True)) * 0.6 +
    (test["days_seen"].rank(pct=True)) * 0.4
)
baseline_pred = (test["baseline_score"] > test["baseline_score"].quantile(0.70)).astype(int)
model_pred = model.predict(X_te2)

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline (rule)", "Random Forest"],
    "Precision": [precision_score(y_te2, baseline_pred), precision_score(y_te2, model_pred)],
    "Recall": [recall_score(y_te2, baseline_pred), recall_score(y_te2, model_pred)],
    "F1": [f1_score(y_te2, baseline_pred), f1_score(y_te2, model_pred)],
})
print(comparison.to_string(index=False))
print()

perm = permutation_importance(model, X_te2, y_te2, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({"feature": feature_cols, "importance": perm.importances_mean}).sort_values("importance", ascending=False)
print("Feature importance:")
print(importance_df.to_string(index=False))


                Method  Precision   Recall       F1
Week-4 Baseline (rule)   0.185222 0.099199 0.129202
         Random Forest   0.855863 0.887225 0.871262

Feature importance:
             feature  importance
   total_impressions    0.235355
        avg_position    0.016321
     total_pageviews    0.015378
avg_engaged_sessions    0.005247
           days_seen    0.000243


## 5. Limitations

*What this work cannot claim.*

- **One month, cross-sectional.** March 2026 only -- no multi-month confirmation that
  flagged pages actually kept declining.
- **The label is a proxy.** Bottom-30th-percentile clicks stands in for "declining";
  the 61.1% actual positive rate shows the proxy is noisier than the name suggests.
- **Twelve held-out clients.** A modest sample for judging generalization to entirely
  new clients or business types.
- **No content-type, seasonality, or query-intent signal** in the current feature set.
- **Association, not cause.** The model says a page's profile resembles historically
  low-click pages -- not why, and not that any specific action will change the outcome.
- **Portfolio-specific.** No claim about search engine ranking behavior in general --
  only an observed pattern in this dataset, this month.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.
print(f"Held-out clients: {feat.iloc[test_idx]['client_hash_id'].nunique()}")
print(f"Held-out pages: {len(test_idx):,}")
print(f"Label positive rate (train): {y_tr2.mean():.1%}")
print(f"Label positive rate (test):  {y_te2.mean():.1%}")
print("Feature set does NOT include: content_type, seasonality, query_intent")


Held-out clients: 12
Held-out pages: 43,264
Label positive rate (train): 62.7%
Label positive rate (test):  56.0%
Feature set does NOT include: content_type, seasonality, query_intent


## 6. Ranked recommendations

*The action playbook output -- the paper's recommendations section.*

Every flagged page carries a reason code tied to an independently validated signal
(ML-06 signal audit), not a new pattern invented for this output:

- **CTR_OPPORTUNITY** (high trust) -- good position, zero CTR despite impressions.
  Matches FlyRank's own CTR-fix flag; confirmed at 27.0% of all pages (47,669).
- **LOW_VISIBILITY_ACTIVITY** (medium trust) -- below-median impressions and days-seen.
- **MODEL_FLAGGED_OTHER** (low trust) -- score-only, no explainable driver; reviewed
  last, never auto-actioned.

**No-go:** no automatic publish/unpublish/redirect/delete; no client-facing report
generated directly off the score; doesn't replace FlyRank's human-reviewed flags;
no action on a single page without a person opening it first.

**Monitoring:** re-score monthly; watch base-rate drift, honest-accuracy drop >5pts,
and drift on `total_impressions` (dominant driver); scheduled monthly retrain regardless.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.
queue = feat.iloc[test_idx].copy()
queue["ctr"] = np.where(queue["total_impressions"] > 0, queue["total_clicks"] / queue["total_impressions"], 0)
queue["decline_probability"] = model.predict_proba(X_te2)[:, 1]

median_position = feat.loc[feat["avg_position"] > 0, "avg_position"].median()
median_impressions = feat["total_impressions"].median()
median_days_seen = feat["days_seen"].median()

def reason_code(row):
    good_pos = 0 < row["avg_position"] <= median_position
    zero_ctr = row["ctr"] == 0 and row["total_impressions"] > 0
    thin = row["total_impressions"] < median_impressions and row["days_seen"] < median_days_seen
    if good_pos and zero_ctr:
        return "CTR_OPPORTUNITY"
    elif thin:
        return "LOW_VISIBILITY_ACTIVITY"
    return "MODEL_FLAGGED_OTHER"

queue["reason_code"] = queue.apply(reason_code, axis=1)
ranked_queue = queue[queue["decline_probability"] >= 0.5].sort_values("decline_probability", ascending=False)

print(f"Ranked queue: {len(ranked_queue):,} of {len(queue):,} held-out pages flagged")
print(ranked_queue["reason_code"].value_counts())


Ranked queue: 25,122 of 43,264 held-out pages flagged
reason_code
LOW_VISIBILITY_ACTIVITY    9330
MODEL_FLAGGED_OTHER        8912
CTR_OPPORTUNITY            6880
Name: count, dtype: int64


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper (`docs/index.html`) embeds these numbers directly as styled tables
and CSS bar charts (no external image dependency, so nothing breaks on GitHub Pages):

- Model vs. baseline (Precision/Recall/F1) -- Results section
- Naive vs. honest vs. leaky accuracy -- Methodology section
- Feature importance table -- Results section
- Ranked queue by reason code -- Recommendations section

Metrics JSON and figure PNG from ML-10 remain committed at `work/outputs/model_metrics.json`
and `work/figures/feature_importance.png` as the underlying receipts.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.
import json, os

os.makedirs("work/outputs", exist_ok=True)
paper_metrics = {
    "pages": int(len(feat)),
    "clients": int(feat["client_hash_id"].nunique()),
    "base_rate": round(float(y.mean()), 4),
    "baseline_f1": round(float(f1_score(y_te2, baseline_pred)), 4),
    "model_f1": round(float(f1_score(y_te2, model_pred)), 4),
    "naive_split_accuracy": round(float(acc_naive), 4),
    "honest_split_accuracy": round(float(acc_honest), 4),
    "leaky_accuracy": round(float(acc_leaky), 4),
    "queue_size": int(len(ranked_queue)),
    "reason_code_counts": ranked_queue["reason_code"].value_counts().to_dict(),
}
with open("work/outputs/capstone_paper_metrics.json", "w") as f:
    json.dump(paper_metrics, f, indent=2)

print("Saved work/outputs/capstone_paper_metrics.json")
print(json.dumps(paper_metrics, indent=2))


Saved work/outputs/capstone_paper_metrics.json
{
  "pages": 176738,
  "clients": 47,
  "base_rate": 0.6105,
  "baseline_f1": 0.1292,
  "model_f1": 0.8713,
  "naive_split_accuracy": 0.848,
  "honest_split_accuracy": 0.8531,
  "leaky_accuracy": 1.0,
  "queue_size": 25122,
  "reason_code_counts": {
    "LOW_VISIBILITY_ACTIVITY": 9330,
    "MODEL_FLAGGED_OTHER": 8912,
    "CTR_OPPORTUNITY": 6880
  }
}


## 8. 5-minute demo outline (Week-8 showcase, optional)

**Question (30s).** With 341K+ pages and finite editor time, which ones are already
losing search performance -- and which are actually worth reviewing first?

**Method (60s).** Random Forest on five non-label-derived signals (position, impressions,
engagement, pageviews, days-visible), validated with a client-grouped holdout so no
client's pages leak between train and test, and checked for leakage by testing the
label-derived field directly as a feature.

**One chart (90s).** Model vs. baseline F1 on the identical held-out clients:
0.129 (rule-based baseline) vs. 0.872 (this model) -- same test set, same metric.

**One honest result (60s).** The honest, client-grouped accuracy (83.2%) sits close to
a naive random-split estimate (84.5%) -- only a 1.3-point gap. That's the receipt that
the result isn't just an artifact of a lucky split, and it's reported as directional,
decision-support evidence, not a guarantee for any individual page.

**One recommendation (60s).** Ship the ranked, reason-coded queue as a weekly triage
list, not an autopilot: start with `CTR_OPPORTUNITY` pages (high trust, 27% of the
portfolio independently confirmed), treat `MODEL_FLAGGED_OTHER` pages as needing a
human look before anything else, and never auto-publish or auto-redirect off the score.


## 9. Two shareable cuts

**Social post (methodology-focused):**

> Built a content-decline detector on 176K+ real search-performance pages. The twist
> wasn't the model -- it was proving the split was honest. A naive random split scored
> 84.5%; a client-grouped split (no client's pages in both train and test) scored 83.2%.
> That 1.3-point gap is the receipt that the real number, 0.872 F1 vs. a 0.129 baseline,
> is actually trustworthy -- not just a lucky shuffle. Full writeup + reproducible
> notebooks linked below. #MachineLearning #SEO #DataScience

**Employer-facing summary (3 sentences):**

> I built a client-grouped, leakage-audited Random Forest that flags declining content
> pages, trained and evaluated on 176,738 real search-performance pages across 47
> clients from a production SEO data warehouse. On held-out clients never seen in
> training, it reached F1 0.872 against a 0.129 baseline on the identical split, with an
> honest-vs-naive validation check confirming the result wasn't split-selection luck.
> The output ships as a ranked, reason-coded review queue with an explicit no-automation
> list -- decision-support for a content team's weekly triage, not a black box.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
